# Multi-Agent Text-to-Image Pipeline
### LangGraph + Llama 3 (Groq) + Stable Diffusion

**Architecture:**
```
User Input
    └── Orchestrator (Llama 3)
            ├── Prompt Engineer (Llama 3)  ─┐
            ├── Style Analyst  (Llama 3)   ─┤── Merger ── Image Generator ── Quality Reviewer
            └── Safety Checker (Llama 3)   ─┘                                      └── END / RETRY
```

**Requirements (requirements.txt):**
```
langgraph>=0.2.0
langchain>=0.2.0
langchain-groq>=0.1.0
langchain-core>=0.2.0
Pillow>=10.0.0
requests>=2.31.0
```

## 1. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q langgraph langchain langchain-groq langchain-core Pillow requests

# Verify installations
import importlib
packages = ['langgraph', 'langchain', 'langchain_groq', 'PIL', 'requests']
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f'  [OK] {pkg}')
    except ImportError:
        print(f'  [MISSING] {pkg} — re-run the cell')

print('\nAll dependencies ready!')

## 2. API Keys & Configuration

**Groq API** (free, fast Llama 3): get your key at https://console.groq.com  
**HuggingFace Token** (optional, for SD 2.1): https://huggingface.co/settings/tokens  

In Colab: go to **Secrets** (key icon in left sidebar) and add:
- `GROQ_API_KEY`  
- `HF_TOKEN` (optional)

In [ ]:
import os

# ── Load from Colab Secrets (recommended) ──────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    HF_TOKEN     = userdata.get('HF_TOKEN') or ''
    print('[Colab Secrets] Keys loaded successfully.')
except Exception:
    # ── Fallback: paste keys directly (not recommended for sharing) ─────────
    GROQ_API_KEY = 'gsk_YOUR_GROQ_KEY_HERE'
    HF_TOKEN     = ''   # leave blank to use pollinations.ai fallback
    print('[Manual] Using hardcoded keys — add to Colab Secrets for safety.')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# ── Image generation backend toggle ────────────────────────────────────────
# True  → HuggingFace Inference API (stabilityai/stable-diffusion-2-1)
# False → pollinations.ai URL API   (no key needed, great for testing)
USE_HF_API = bool(HF_TOKEN)

# ── Llama 3 model via Groq ──────────────────────────────────────────────────
GROQ_MODEL = 'llama3-8b-8192'

# ── Quality reviewer threshold ──────────────────────────────────────────────
QUALITY_PASS_SCORE = 7   # score out of 10
MAX_RETRIES        = 2

print(f'\nConfig summary:')
print(f'  LLM model    : {GROQ_MODEL}')
print(f'  Image backend: {"HuggingFace SD 2.1" if USE_HF_API else "pollinations.ai (no key)"}')
print(f'  Quality gate : score >= {QUALITY_PASS_SCORE}/10 or max {MAX_RETRIES} retries')

## 3. Define AgentState

The `AgentState` TypedDict is the shared memory that flows through every node in the graph. Each node reads from it and writes its output back into it.

In [ ]:
from typing import TypedDict, Optional, Annotated
import operator

class AgentState(TypedDict):
    """Shared state passed between all LangGraph nodes."""
    user_input      : str                    # raw text from the user
    enhanced_prompt : str                    # prompt_engineer output
    style_tags      : str                    # style_analyst output
    final_prompt    : str                    # merger output (sent to image gen)
    image_url       : str                    # URL or base64 of generated image
    review_score    : int                    # quality_reviewer score (0-10)
    retry_count     : int                    # number of retries so far
    safe            : bool                   # safety_checker result
    error_message   : Optional[str]          # set if pipeline aborted early

print('AgentState fields:')
for field, ftype in AgentState.__annotations__.items():
    print(f'  {field:<20} {str(ftype)}')

## 4. Define Agent Node Functions

Each function is a LangGraph node. It receives the current `AgentState`, runs its Llama 3 chain, and returns a **partial state dict** with only the fields it updates.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Shared LLM instance (all agents share the same model, different prompts) ──
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.7,
    api_key=GROQ_API_KEY
)

parser = StrOutputParser()

def _run_chain(system_prompt: str, human_message: str) -> str:
    """Helper: build a prompt template, run it, return the string output."""
    prompt = ChatPromptTemplate.from_messages([
        ('system', system_prompt),
        ('human',  '{input}')
    ])
    chain = prompt | llm | parser
    return chain.invoke({'input': human_message}).strip()

print('LLM and chain helper initialised.')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# NODE 1 — Orchestrator
# Decides whether the input is suitable and routes it into the pipeline.
# ────────────────────────────────────────────────────────────────────────────

ORCHESTRATOR_SYSTEM = """You are an orchestrator agent for a text-to-image AI pipeline.
Your job is to briefly acknowledge the user's request, confirm what type of image
they want, and output a clean, normalised version of their text (fixing typos,
clarifying ambiguities). Output ONLY the normalised text, nothing else."""

def orchestrator(state: AgentState) -> dict:
    print('\n[ORCHESTRATOR] Processing user input...')
    normalised = _run_chain(
        ORCHESTRATOR_SYSTEM,
        f'User input: {state["user_input"]}'
    )
    print(f'  Normalised input: {normalised[:120]}...' if len(normalised) > 120 else f'  Normalised: {normalised}')
    # Initialise retry counter here
    return {
        'user_input'   : normalised,
        'retry_count'  : state.get('retry_count', 0),
        'safe'         : True,
        'error_message': None
    }

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# NODE 2 — Safety Checker
# Uses Llama 3 with a strict safety system prompt (Llama Guard style).
# ────────────────────────────────────────────────────────────────────────────

SAFETY_SYSTEM = """You are a content safety classifier.
Analyse the user's image request. Reply with ONLY one word:
- SAFE   if the request is appropriate (landscapes, objects, people in normal contexts, fantasy, sci-fi, etc.)
- UNSAFE if the request contains violence, explicit sexual content, hate speech, illegal activities, or self-harm.
No explanation. Just SAFE or UNSAFE."""

def safety_checker(state: AgentState) -> dict:
    print('\n[SAFETY CHECKER] Evaluating content...')
    verdict = _run_chain(
        SAFETY_SYSTEM,
        state['user_input']
    ).upper().strip()

    is_safe = 'UNSAFE' not in verdict
    print(f'  Verdict: {"SAFE" if is_safe else "UNSAFE"}')

    if not is_safe:
        return {
            'safe'         : False,
            'error_message': 'Content flagged as unsafe by safety checker. Pipeline aborted.'
        }
    return {'safe': True}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# NODE 3 — Prompt Engineer
# Rewrites the raw user text into a rich, detailed image generation prompt.
# ────────────────────────────────────────────────────────────────────────────

PROMPT_ENGINEER_SYSTEM = """You are an expert image prompt engineer for Stable Diffusion.
Transform the user's simple description into a rich, detailed image generation prompt.
Include: subject details, environment, textures, materials, composition.
Do NOT include style, lighting, or camera details — those come from another agent.
Output ONLY the enhanced prompt text (2-4 sentences). No preamble."""

def prompt_engineer(state: AgentState) -> dict:
    print('\n[PROMPT ENGINEER] Enhancing prompt...')
    enhanced = _run_chain(
        PROMPT_ENGINEER_SYSTEM,
        f'Transform this into a rich image prompt: {state["user_input"]}'
    )
    print(f'  Enhanced: {enhanced[:150]}...' if len(enhanced) > 150 else f'  Enhanced: {enhanced}')
    return {'enhanced_prompt': enhanced}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# NODE 4 — Style Analyst
# Generates style, lighting, mood, and camera tags to append to the prompt.
# ────────────────────────────────────────────────────────────────────────────

STYLE_ANALYST_SYSTEM = """You are an art direction specialist for AI image generation.
Given an image description, output ONLY a comma-separated list of style and technical keywords.
Include: art style (e.g. photorealistic, oil painting, concept art), lighting (e.g. golden hour,
dramatic backlight), mood (e.g. serene, epic, melancholic), camera (e.g. wide angle, macro,
35mm lens). Output 8-12 keywords, comma-separated, no sentences, no preamble."""

def style_analyst(state: AgentState) -> dict:
    print('\n[STYLE ANALYST] Generating style tags...')
    tags = _run_chain(
        STYLE_ANALYST_SYSTEM,
        f'Generate style tags for: {state["user_input"]}'
    )
    print(f'  Style tags: {tags}')
    return {'style_tags': tags}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# NODE 5 — Merger
# Combines enhanced_prompt + style_tags into the final prompt for image gen.
# ────────────────────────────────────────────────────────────────────────────

def merger(state: AgentState) -> dict:
    print('\n[MERGER] Combining prompt + style tags...')
    enhanced = state.get('enhanced_prompt', state['user_input'])
    style    = state.get('style_tags', '')
    final    = f'{enhanced}, {style}' if style else enhanced
    print(f'  Final prompt ({len(final)} chars): {final[:120]}...' if len(final) > 120 else f'  Final prompt: {final}')
    return {'final_prompt': final}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# NODE 7 — Quality Reviewer
# Llama 3 scores the final prompt (simulating a review of the image intent).
# In a production system, this would use a vision model on the actual image.
# ────────────────────────────────────────────────────────────────────────────

REVIEWER_SYSTEM = """You are a quality reviewer for AI-generated image prompts.
Score the final image prompt on a scale of 1-10 based on:
- Clarity and specificity (0-3 points)
- Visual richness and detail (0-3 points)
- Style and technical quality keywords (0-2 points)
- Overall coherence (0-2 points)
Output ONLY a single integer between 1 and 10. Nothing else."""

def quality_reviewer(state: AgentState) -> dict:
    print('\n[QUALITY REVIEWER] Scoring the final prompt...')
    raw_score = _run_chain(
        REVIEWER_SYSTEM,
        f'Score this image prompt: {state["final_prompt"]}'
    )
    # Safely extract integer from the response
    import re
    numbers = re.findall(r'\b(\d{1,2})\b', raw_score)
    score = int(numbers[0]) if numbers else 5
    score = max(1, min(10, score))  # clamp to 1-10

    new_retry = state.get('retry_count', 0)
    if score < QUALITY_PASS_SCORE:
        new_retry += 1

    print(f'  Score: {score}/10  |  Retry count: {new_retry}/{MAX_RETRIES}')
    action = 'PASS' if (score >= QUALITY_PASS_SCORE or new_retry >= MAX_RETRIES) else 'RETRY'
    print(f'  Decision: {action}')

    return {
        'review_score': score,
        'retry_count' : new_retry
    }

## 6. Image Generation Tool

Two backends available:
- **HuggingFace Inference API** — `stabilityai/stable-diffusion-2-1` (free tier, needs HF token)
- **pollinations.ai** — zero-setup, no API key, great for testing

In [ ]:
import requests
import io
import base64
from PIL import Image
from IPython.display import display as ipy_display, Image as IPyImage

HF_API_URL = 'https://api-inference.huggingface.co/models/stabilityai/stable-diffusion-2-1'

def generate_image_hf(prompt: str) -> str:
    """Generate image via HuggingFace Inference API. Returns base64 data URI."""
    print(f'  [HF API] Sending prompt to SD 2.1...')
    headers = {'Authorization': f'Bearer {HF_TOKEN}'}
    payload = {'inputs': prompt, 'parameters': {'num_inference_steps': 30}}
    response = requests.post(HF_API_URL, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    image_bytes = response.content
    img = Image.open(io.BytesIO(image_bytes))
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'data:image/png;base64,{b64}'

def generate_image_pollinations(prompt: str) -> str:
    """Generate image via pollinations.ai. Returns the direct image URL."""
    import urllib.parse
    encoded = urllib.parse.quote(prompt[:500])  # URL-safe, truncate if needed
    url = f'https://image.pollinations.ai/prompt/{encoded}?width=768&height=768&nologo=true'
    print(f'  [pollinations.ai] Requesting image...')
    # Verify the URL returns an image
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return url

# ────────────────────────────────────────────────────────────────────────────
# NODE 6 — Image Generator (the LangGraph node)
# ────────────────────────────────────────────────────────────────────────────

def image_generator(state: AgentState) -> dict:
    print('\n[IMAGE GENERATOR] Generating image...')
    prompt = state.get('final_prompt') or state['user_input']
    try:
        if USE_HF_API:
            url = generate_image_hf(prompt)
        else:
            url = generate_image_pollinations(prompt)
        print(f'  Image ready: {url[:80]}...' if len(url) > 80 else f'  Image URL: {url}')
        return {'image_url': url}
    except Exception as e:
        print(f'  [ERROR] Image generation failed: {e}')
        # Return a placeholder so the pipeline can continue
        return {'image_url': '', 'error_message': f'Image generation error: {str(e)}'}

print('Image generation nodes defined.')
print(f'Active backend: {"HuggingFace SD 2.1" if USE_HF_API else "pollinations.ai"}')

## 5. Build & Compile the LangGraph

Here we wire all nodes together, add conditional edges for the safety check and quality retry loop, and compile the graph.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.constants import START

# ── Routing functions (used in conditional_edges) ────────────────────────────

def route_after_safety(state: AgentState) -> str:
    """After safety check: go to parallel workers if safe, else END."""
    if not state.get('safe', True):
        print('  [ROUTER] Unsafe content — routing to END.')
        return 'unsafe'
    return 'safe'

def route_after_review(state: AgentState) -> str:
    """After quality review: PASS (end) or RETRY (back to prompt engineer)."""
    score      = state.get('review_score', 0)
    retries    = state.get('retry_count', 0)
    if score >= QUALITY_PASS_SCORE or retries >= MAX_RETRIES:
        print(f'  [ROUTER] Quality PASSED (score={score}, retries={retries}) → END')
        return 'pass'
    print(f'  [ROUTER] Quality LOW (score={score}) → RETRY (attempt {retries}/{MAX_RETRIES})')
    return 'retry'

# ── Build the graph ────────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Add all nodes
builder.add_node('orchestrator',    orchestrator)
builder.add_node('safety_checker',  safety_checker)
builder.add_node('prompt_engineer', prompt_engineer)
builder.add_node('style_analyst',   style_analyst)
builder.add_node('merger',          merger)
builder.add_node('image_generator', image_generator)
builder.add_node('quality_reviewer',quality_reviewer)

# Entry point
builder.add_edge(START, 'orchestrator')

# Orchestrator → Safety
builder.add_edge('orchestrator', 'safety_checker')

# Safety → (parallel fan-out OR end)
builder.add_conditional_edges(
    'safety_checker',
    route_after_safety,
    {
        'safe'  : 'prompt_engineer',   # start parallel fan-out
        'unsafe': END
    }
)

# NOTE: True Send()-based parallelism requires langgraph >=0.2 with async support.
# Here we use sequential fan-out (prompt_engineer → style_analyst) which is
# compatible with all Colab environments. Both run before merger.
builder.add_edge('prompt_engineer', 'style_analyst')
builder.add_edge('style_analyst',   'merger')

# Merger → Image gen → Review
builder.add_edge('merger',          'image_generator')
builder.add_edge('image_generator', 'quality_reviewer')

# Review → (pass to END or retry from prompt_engineer)
builder.add_conditional_edges(
    'quality_reviewer',
    route_after_review,
    {
        'pass' : END,
        'retry': 'prompt_engineer'
    }
)

# Compile
graph = builder.compile()

print('Graph compiled successfully!')

In [ ]:
# ── Visualise the graph structure ─────────────────────────────────────────────
print('=== GRAPH STRUCTURE (Mermaid) ===')
try:
    print(graph.get_graph().draw_mermaid())
except Exception:
    print('(Mermaid not available — showing ASCII)')
    try:
        graph.get_graph().print_ascii()
    except Exception as e:
        print(f'(ASCII also unavailable: {e})')

print('\nNodes:', list(graph.get_graph().nodes.keys()))

## 7. Run Pipeline End-to-End (Demo)

Two example prompts. Watch the verbose trace as each agent node executes.

In [ ]:
def run_pipeline(user_text: str) -> AgentState:
    """Run the full multi-agent pipeline and return the final state."""
    print('=' * 60)
    print(f'INPUT: "{user_text}"')
    print('=' * 60)

    initial_state: AgentState = {
        'user_input'     : user_text,
        'enhanced_prompt': '',
        'style_tags'     : '',
        'final_prompt'   : '',
        'image_url'      : '',
        'review_score'   : 0,
        'retry_count'    : 0,
        'safe'           : True,
        'error_message'  : None
    }

    try:
        final_state = graph.invoke(initial_state)
    except Exception as e:
        print(f'\n[PIPELINE ERROR] {e}')
        initial_state['error_message'] = str(e)
        return initial_state

    print('\n' + '=' * 60)
    print('PIPELINE COMPLETE')
    print('=' * 60)
    print(f'  Final prompt  : {final_state.get("final_prompt", "")[:120]}')
    print(f'  Quality score : {final_state.get("review_score", 0)}/10')
    print(f'  Retry count   : {final_state.get("retry_count", 0)}')
    if final_state.get('error_message'):
        print(f'  Error         : {final_state["error_message"]}')
    return final_state

In [ ]:
# ── Demo 1: Landscape scene ───────────────────────────────────────────────────
result1 = run_pipeline('a futuristic city at sunset with flying cars')

In [ ]:
# ── Demo 2: Fantasy character ─────────────────────────────────────────────────
result2 = run_pipeline('an ancient wizard in a dark forest holding a glowing staff')

## 8. Display Output Images

Render the generated images inline in the notebook.

In [ ]:
from IPython.display import display as ipy_display, Image as IPyImage, HTML
import io, base64, requests
from PIL import Image

def show_result(state: AgentState, title: str = ''):
    """Display the image and metadata from a pipeline result."""
    if title:
        ipy_display(HTML(f'<h3>{title}</h3>'))

    if state.get('error_message') and not state.get('image_url'):
        ipy_display(HTML(f'<p style="color:red">Pipeline error: {state["error_message"]}</p>'))
        return

    if not state.get('image_url'):
        ipy_display(HTML('<p style="color:orange">No image generated.</p>'))
        return

    url = state['image_url']

    try:
        if url.startswith('data:image'):
            # base64 data URI from HF API
            header, b64data = url.split(',', 1)
            img_bytes = base64.b64decode(b64data)
        else:
            # Regular URL (pollinations.ai)
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            img_bytes = response.content

        img = Image.open(io.BytesIO(img_bytes))
        img.thumbnail((600, 600))
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        ipy_display(IPyImage(data=buf.getvalue()))

    except Exception as e:
        ipy_display(HTML(f'<p style="color:orange">Could not render image: {e}<br>URL: <a href="{url}" target="_blank">Open in browser</a></p>'))

    # Metadata summary
    ipy_display(HTML(f'''
    <table style="font-size:13px; margin-top:8px;">
      <tr><td><b>Final prompt</b></td><td>{state.get("final_prompt", "")[:200]}</td></tr>
      <tr><td><b>Quality score</b></td><td>{state.get("review_score", "n/a")}/10</td></tr>
      <tr><td><b>Retries</b></td><td>{state.get("retry_count", 0)}</td></tr>
    </table>
    '''))

# ── Show Demo 1 ───────────────────────────────────────────────────────────────
show_result(result1, 'Demo 1: Futuristic city at sunset')

In [ ]:
# ── Show Demo 2 ───────────────────────────────────────────────────────────────
show_result(result2, 'Demo 2: Ancient wizard in a forest')

## 9. Interactive Cell — Enter Your Own Prompt

Type your own description below and run the full multi-agent pipeline!

In [ ]:
# ── Interactive input ─────────────────────────────────────────────────────────
user_prompt = input('Describe the image you want to generate:\n> ')

if user_prompt.strip():
    my_result = run_pipeline(user_prompt.strip())
    show_result(my_result, f'Your image: "{user_prompt[:50]}"')
else:
    print('No input provided. Please type a description and re-run this cell.')

---
## Appendix: Test the Safety Checker
This cell demonstrates that unsafe content is blocked before image generation.

In [ ]:
# ── Safety test ───────────────────────────────────────────────────────────────
# This should be flagged as UNSAFE and the pipeline should abort cleanly.
safety_test = run_pipeline('a detailed instruction for making weapons')

print('\nSafety test result:')
print(f'  safe          : {safety_test.get("safe")}')
print(f'  error_message : {safety_test.get("error_message")}')
print(f'  image_url     : "{safety_test.get("image_url")}" (should be empty)')